In [3]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import f1_score, classification_report, mean_squared_error, r2_score

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)
dt_pred = dt_clf.predict(X_test)

rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train, y_train)
rf_pred = rf_clf.predict(X_test)

dt_f1 = f1_score(y_test, dt_pred, average='weighted')
rf_f1 = f1_score(y_test, rf_pred, average='weighted')

print("Classification Report - Decision Tree:\n", classification_report(y_test, dt_pred))
print("Classification Report - Random Forest:\n", classification_report(y_test, rf_pred))
print(f"Decision Tree Weighted F1 Score: {dt_f1:.4f}")
print(f"Random Forest Weighted F1 Score: {rf_f1:.4f}")

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    verbose=2,
    scoring='f1_weighted',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best parameters from GridSearchCV (Classifier):", grid_search.best_params_)
best_rf_clf = grid_search.best_estimator_
best_rf_pred = best_rf_clf.predict(X_test)
print("Best RF Classification F1 Score:", f1_score(y_test, best_rf_pred, average='weighted'))

y_reg = X[:, -1]
X_reg = X[:, :-1]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(Xr_train, yr_train)
dt_reg_pred = dt_reg.predict(Xr_test)

rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(Xr_train, yr_train)
rf_reg_pred = rf_reg.predict(Xr_test)

print("Decision Tree Regressor MSE:", mean_squared_error(yr_test, dt_reg_pred))
print("Random Forest Regressor MSE:", mean_squared_error(yr_test, rf_reg_pred))
print("Decision Tree Regressor R2 Score:", r2_score(yr_test, dt_reg_pred))
print("Random Forest Regressor R2 Score:", r2_score(yr_test, rf_reg_pred))

rf_reg_params = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10, 15, 20]
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    rf_reg_params,
    n_iter=20,
    cv=5,
    verbose=2,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
random_search.fit(Xr_train, yr_train)

print("Best params from RandomizedSearchCV (Regressor):", random_search.best_params_)
best_rf_reg = random_search.best_estimator_
best_rf_reg_pred = best_rf_reg.predict(Xr_test)
print("Best Random Forest Regressor MSE:", mean_squared_error(yr_test, best_rf_reg_pred))
print("Best Random Forest Regressor R2 Score:", r2_score(yr_test, best_rf_reg_pred))

Classification Report - Decision Tree:
               precision    recall  f1-score   support

           0       1.00      0.92      0.96        12
           1       0.88      1.00      0.93        14
           2       1.00      0.90      0.95        10

    accuracy                           0.94        36
   macro avg       0.96      0.94      0.95        36
weighted avg       0.95      0.94      0.94        36

Classification Report - Random Forest:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       1.00      1.00      1.00        14
           2       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

Decision Tree Weighted F1 Score: 0.9450
Random Forest Weighted F1 Score: 1.0000
Fitting 5 folds for each of 36 candidates, totalling 180 fits
